# compiler-opt: multi-suite benchmark battery (Colab Pro)

One suite per notebook tab. Set the form below, then **Runtime → Run all**.
With Colab Pro, enable **Runtime → Background execution** so the sweep
survives a closed tab.

- Interruption-proof: each program writes its own JSON; a rerun skips
  finished programs, so just Run all again after any disconnect.
- Results sync to `Drive/compiler-opt-colab/results/` every 2 minutes.
- CPU runtime is all this needs — do **not** pick a GPU.

In [ ]:
SUITE = "mibench-v1"  # @param ["mibench-v1", "chstone-v0", "npb-v0", "blas-v0", "opencv-v0", "poj104-v1", "jotaibench-v0", "csmith-v0", "cbench-v1"]
SAMPLE_N = 0  # @param {type:"integer"}  (0 = whole suite; use 50-100 for big/generator suites)
SEED = 42  # @param {type:"integer"}
REPO_URL = "https://github.com/dimitrijepesic/compiler-opt"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}

import os
os.environ.update({"SUITE": SUITE, "SAMPLE_N": str(SAMPLE_N),
                   "SEED": str(SEED), "REPO_URL": REPO_URL, "BRANCH": BRANCH})
print(f"suite={SUITE} sample_n={SAMPLE_N} seed={SEED}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/compiler-opt-colab/results
# Restore CompilerGym runtime+dataset cache from Drive if a previous session saved it
!if [ -f /content/drive/MyDrive/compiler-opt-colab/cgym_cache.tar ]; then \
   tar -xf /content/drive/MyDrive/compiler-opt-colab/cgym_cache.tar -C ~ && echo 'cache restored'; \
 else echo 'no cache yet (first session downloads fresh)'; fi

In [ ]:
%%bash
# Environment setup (~8 min first session). Colab is Ubuntu 22.04: system
# python3.10 + libtinfo5 are exactly what CompilerGym 0.2.5 needs.
set -e
sudo apt-get -qq update && sudo apt-get -qq install -y python3.10-venv libtinfo5 build-essential > /dev/null
[ -d /content/compiler-opt ] || git clone --branch "$BRANCH" "$REPO_URL" /content/compiler-opt
cd /content/compiler-opt && git pull --ff-only
if [ ! -d /content/venv-cgym ]; then
  /usr/bin/python3.10 -m venv /content/venv-cgym
  source /content/venv-cgym/bin/activate
  pip -q install "pip<24.1" setuptools==65.5.0 wheel==0.38.4
  pip -q install gym==0.21.0
  pip -q install compiler-gym==0.2.5 "numpy<2" pyyaml tqdm scipy matplotlib
  pip -q install torch --index-url https://download.pytorch.org/whl/cpu
  pip -q install torch-geometric
fi
source /content/venv-cgym/bin/activate
python -c "import compiler_gym; print('CompilerGym OK', compiler_gym.__version__)"

In [ ]:
%%bash
# Launch the battery detached; the next cell monitors and syncs to Drive.
cd /content/compiler-opt
source /content/venv-cgym/bin/activate
EXTRA=""
[ "$SAMPLE_N" != "0" ] && EXTRA="--sample-n $SAMPLE_N"
nohup python scripts/benchmark_battery.py --suite "$SUITE" --seed "$SEED" $EXTRA \
  > "/content/battery_${SUITE}.log" 2>&1 &
echo "launched: suite=$SUITE"

In [ ]:
# Monitor + sync loop: prints progress, rsyncs results to Drive every 2 min,
# exits when the battery finishes. Safe to re-run after a disconnect.
import os, subprocess, time
suite = os.environ['SUITE']
log = f"/content/battery_{suite}.log"
def running():
    return subprocess.run(['pgrep', '-f', 'benchmark_battery.py'],
                          capture_output=True).returncode == 0
while True:
    subprocess.run(['rsync', '-a', '/content/compiler-opt/results/battery/',
                    '/content/drive/MyDrive/compiler-opt-colab/results/battery/'])
    out = subprocess.run(['tail', '-3', log], capture_output=True, text=True).stdout
    print(out.strip()[-300:], flush=True)
    if not running():
        print('battery process finished')
        break
    time.sleep(120)

In [ ]:
%%bash
# Final sync + save the CompilerGym cache for future sessions (first run only)
rsync -a /content/compiler-opt/results/battery/ /content/drive/MyDrive/compiler-opt-colab/results/battery/
if [ ! -f /content/drive/MyDrive/compiler-opt-colab/cgym_cache.tar ]; then
  tar -cf /content/drive/MyDrive/compiler-opt-colab/cgym_cache.tar \
    -C ~ .local/share/compiler_gym .cache/compiler_gym 2>/dev/null || true
  echo 'cache saved to Drive'
fi
tail -5 "/content/battery_${SUITE}.log"
echo ALL-DONE